# Extract DINOv2 visual features

Run once per (dataset, seed). `run_al_sampler.ipynb` reuses these caches, so the
90k-image forward pass happens once instead of once per sampler.

Seed matters: ImageFolder datasets (histoset, skintissue) are split by a seeded
generator, so a cache is only valid for the seed it was built with.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASETS = ["pathmnist"]
SEEDS = [42]
FEATURE_DIR = "/kaggle/working/features"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

from data.loaders import get_data_loaders, get_sample_ids
from data.identity import sample_order_fingerprint
from features.visual import get_or_extract_features

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

DEVICE = torch.device(config["device"])
VIT_NAME = config.get("models", {}).get("vit", "facebook/dinov2-base")
assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
missing = {name: DATA_PATHS[name] for name in DATASETS if not Path(DATA_PATHS[name]).exists()}
assert not missing, f"Missing Kaggle inputs: {missing}"
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE} backbone={VIT_NAME} cache={FEATURE_DIR}")

In [ ]:
for seed in SEEDS:
    for dataset in DATASETS:
        print("=" * 60)
        print(f"{dataset} | seed {seed}")
        train_loader, test_loader, _ = get_data_loaders(DATA_PATHS[dataset], seed, verbose=True)
        train_features, test_features = get_or_extract_features(
            train_loader, test_loader, dataset, seed, VIT_NAME, DEVICE,
            cache_dir=FEATURE_DIR,
            train_fingerprint=sample_order_fingerprint(get_sample_ids(train_loader.dataset)),
            test_fingerprint=sample_order_fingerprint(get_sample_ids(test_loader.dataset)),
        )
        print(f"  train {train_features.shape} | test {test_features.shape}")

In [ ]:
# Zip everything this notebook produced so it downloads as one file.
import shutil

SOURCE = Path('/kaggle/working/features')
ARCHIVE = Path("/kaggle/working/visual_features")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6
print(f"{ARCHIVE}.zip  ({size_mb:.1f} MB)")